In [2]:
!pip install -q ultralytics

In [3]:
from ultralytics import YOLO
import numpy as np

MODEL_PATH = "best.pt"      # 최종 모델이 나오면 파일만 교체
NAMES = {0: "균열", 1: "포트홀"}
_model = None


def _get_model():
    global _model
    if _model is None:
        _model = YOLO(MODEL_PATH)
    return _model


def _poly_area(xy):
    x, y = xy[:, 0], xy[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def detect_damage(image_path, conf=0.25):
    """사진 경로를 넣으면 (검출 목록, 표시용 이미지)를 반환한다.

    검출 목록: [{'종류','신뢰도','박스':[x1,y1,x2,y2],'면적비율(%)'}, ...]
    표시용 이미지: 검출 결과가 그려진 RGB numpy 배열 (웹에 바로 표시 가능)
    """
    r = _get_model().predict(image_path, imgsz=1280, conf=conf, verbose=False)[0]
    h, w = r.orig_shape
    out = []
    for i, box in enumerate(r.boxes):
        area = 0.0
        if r.masks is not None:
            area = _poly_area(r.masks.xy[i]) / (w * h) * 100
        out.append({
            "종류": NAMES[int(box.cls[0])],
            "신뢰도": round(float(box.conf[0]), 3),
            "박스": [round(float(v), 1) for v in box.xyxy[0]],
            "면적비율(%)": round(float(area), 2),
        })
    annotated = r.plot()[..., ::-1]   # BGR -> RGB
    return out, annotated

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
